# COVER SHAZAM
This version of the algorithm builds on the previous work to enable detection for covers and alternative versions.

*Note: This notebook is less structured than others and allows for execution of large blocks of code all at once*

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter
from collections import defaultdict
import subprocess
import sys
import os

In [ ]:
CHROMA_LABELS = ['C', 'C#', 'D', 'D#', 'E', 'F',
                 'F#', 'G', 'G#', 'A', 'A#', 'B']


class ChromaShazam:
    """
    Cover-song-friendly audio fingerprinting.
    Chroma + source separation + triplet hashing.
    """

    def __init__(self, sample_rate=22050, hop_length=512):
        self.sample_rate = sample_rate
        self.hop_length = hop_length
        self.database = {}
        self.song_names = {}
        self.database_info = {}

    # 1. Load audio
    def load_audio(self, filepath):
        audio, _ = librosa.load(filepath, sr=self.sample_rate, mono=True)
        return audio

    # 2. Source separation
    def separate_vocals(self, filepath, output_dir="separated"):
        cmd = [
            sys.executable, "-m", "demucs",
            "--two-stems", "vocals",
            "-n", "htdemucs",
            "-o", output_dir,
            filepath
        ]

        print(f"  → Running Demucs...")
        result = subprocess.run(
            cmd, check=False, capture_output=True, text=True
        )

        if result.returncode != 0:
            print("  Demucs failed!")
            print("  STDERR:", result.stderr)
            raise RuntimeError(f"Demucs failed on {filepath}")

        track_name = os.path.splitext(os.path.basename(filepath))[0]
        vocal_path = os.path.join(
            output_dir, "htdemucs", track_name, "vocals.wav"
        )

        if not os.path.exists(vocal_path):
            htdemucs_dir = os.path.join(output_dir, "htdemucs")
            if os.path.exists(htdemucs_dir):
                for folder in os.listdir(htdemucs_dir):
                    candidate = os.path.join(htdemucs_dir, folder, "vocals.wav")
                    if os.path.exists(candidate):
                        vocal_path = candidate
                        break

            if not os.path.exists(vocal_path):
                raise FileNotFoundError(f"Could not find vocals.wav")

        vocal_audio, _ = librosa.load(
            vocal_path, sr=self.sample_rate, mono=True
        )

        print(f"  → Separated vocals: {len(vocal_audio)/self.sample_rate:.2f}s")
        return vocal_audio

    # 3. Compute chromagram (simple, deterministic, no beat tracking)
    def compute_chromagram(self, audio):
        """
        Plain CQT chromagram. One frame per hop. Fully deterministic.
        """
        chroma = librosa.feature.chroma_cqt(
            y=audio,
            sr=self.sample_rate,
            hop_length=self.hop_length,
            norm=2
        )

        times = librosa.frames_to_time(
            np.arange(chroma.shape[1]),
            sr=self.sample_rate,
            hop_length=self.hop_length
        )

        return chroma, times

    # 4. Find peaks
    def find_peaks(self, chroma, threshold=0.15):
        neighborhood_size = (5, 30)
        local_max = maximum_filter(chroma, size=neighborhood_size)

        peaks_mask = (chroma == local_max) & (chroma > threshold)
        peak_coords = np.argwhere(peaks_mask)

        peak_amplitudes = chroma[peaks_mask]
        sorted_indices = np.argsort(peak_amplitudes)[::-1]
        peak_coords = peak_coords[sorted_indices]

        return peak_coords

    # 5. Create constellation map
    def create_constellation_map(self, audio, max_peaks=500, plot=True):
        chroma, times = self.compute_chromagram(audio)
        peaks = self.find_peaks(chroma, threshold=0.15)

        if len(peaks) > max_peaks:
            peaks = peaks[:max_peaks]

        constellation = [(int(p[1]), int(p[0])) for p in peaks]

        if plot:
            self.plot_constellation(chroma, constellation, times)

        return constellation, times, chroma

    def plot_constellation(self, chroma, constellation, times):
        fig, (ax_chroma, ax_const) = plt.subplots(1, 2, figsize=(14, 5))

        # interpolation='nearest' stops the blurring
        img = ax_chroma.imshow(
            chroma,
            aspect='auto',
            origin='lower',
            extent=[times[0], times[-1], -0.5, 11.5],
            cmap='magma',
            interpolation='nearest'    # ← this is the fix
        )
        ax_chroma.set_yticks(range(12))
        ax_chroma.set_yticklabels(CHROMA_LABELS)
        ax_chroma.set_xlabel('Time (s)')
        ax_chroma.set_ylabel('Pitch class')
        ax_chroma.set_title('Chromagram')
        fig.colorbar(img, ax=ax_chroma)

        # add horizontal grid lines between pitch classes
        for i in range(12):
            ax_chroma.axhline(y=i - 0.5, color='white', linewidth=0.3, alpha=0.5)

        # Constellation map (same fix)
        if constellation:
            time_indices, chroma_indices = zip(*constellation)
            ax_const.scatter(
                [times[t] for t in time_indices],
                list(chroma_indices),
                c='red', s=20, marker='x', alpha=0.6
            )

        ax_const.set_yticks(range(12))
        ax_const.set_yticklabels(CHROMA_LABELS)
        ax_const.set_ylim(-0.5, 11.5)
        ax_const.set_xlabel('Time (s)')
        ax_const.set_ylabel('Pitch class')
        ax_const.set_title(f'Constellation Map ({len(constellation)} peaks)')
        ax_const.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    # 7. Generate fingerprints (triplets + moderate quantization)
    def generate_fingerprints(
        self,
        constellation,
        fan_value=15,
        delta_t_min=5,
        delta_t_max=200,
        delta_f_max=6,
        dt_quantize=5      # ~0.12 seconds per bucket
    ):
        """
        Triplet hashing with moderately quantized delta_t.

        Hash: (c1, c2, c3, dt1 // 5, dt2 // 5)
        Space: 12 × 12 × 12 × 40 × 40 ≈ 2.76 million
        """
        fingerprints = []
        constellation_sorted = sorted(constellation, key=lambda x: x[0])
        n = len(constellation_sorted)

        for i in range(n):
            t1, c1 = constellation_sorted[i]

            for j in range(i + 1, min(i + fan_value + 1, n)):
                t2, c2 = constellation_sorted[j]
                dt1 = t2 - t1
                dc1 = abs(c2 - c1)
                dc1 = min(dc1, 12 - dc1)

                if not (delta_t_min <= dt1 <= delta_t_max and dc1 <= delta_f_max):
                    continue

                for k in range(j + 1, min(j + fan_value + 1, n)):
                    t3, c3 = constellation_sorted[k]
                    dt2 = t3 - t2
                    dc2 = abs(c3 - c2)
                    dc2 = min(dc2, 12 - dc2)

                    if not (delta_t_min <= dt2 <= delta_t_max and dc2 <= delta_f_max):
                        continue

                    hash_value = (c1, c2, c3,
                                  dt1 // dt_quantize,
                                  dt2 // dt_quantize)
                    fingerprints.append((hash_value, t1))

        return fingerprints

    # 8. Add song to database
    def add_song_to_database_with_separation(
        self, filepath, song_id, max_peaks=500
    ):
        print(f"Processing (with separation): {song_id}")

        vocal_audio = self.separate_vocals(filepath)

        constellation, times, chroma = self.create_constellation_map(
            vocal_audio, max_peaks=max_peaks, plot=True
        )

        fingerprints = self.generate_fingerprints(constellation)

        self.database[song_id] = fingerprints
        self.song_names[song_id] = os.path.basename(filepath)
        self.database_info[song_id] = {
            'n_fingerprints': len(fingerprints),
            'n_peaks': len(constellation),
            'duration': len(vocal_audio) / self.sample_rate
        }

        print(f"  → Generated {len(fingerprints)} fingerprints "
              f"from {len(constellation)} peaks")
        print()

    # 9. Match audio
    def match_audio(self, audio_clip, plot_results=True):
        """
        Match query against database.
        IDF-weighted, length-normalized, slope-penalized.
        """
        constellation, times, chroma = self.create_constellation_map(
            audio_clip, max_peaks=500, plot=plot_results
        )

        query_fingerprints = self.generate_fingerprints(constellation)
        n_query_fps = len(query_fingerprints)
        print(f"Query has {n_query_fps} fingerprints from "
              f"{len(constellation)} peaks")

        # Hash -> time lookup for query
        query_hashes = {}
        for fp_hash, fp_time in query_fingerprints:
            query_hashes.setdefault(fp_hash, []).append(fp_time)

        # IDF weights
        hash_song_count = defaultdict(set)
        for song_id, db_fps in self.database.items():
            for db_hash, _ in db_fps:
                hash_song_count[db_hash].add(song_id)

        n_songs = max(len(self.database), 1)

        def idf_weight(h):
            n_containing = len(hash_song_count.get(h, set()))
            if n_containing == 0:
                return 0.0
            return np.log(n_songs / n_containing) + 1.0

        # Collect matches
        matches = defaultdict(list)
        for song_id, db_fps in self.database.items():
            for db_hash, db_time in db_fps:
                if db_hash in query_hashes:
                    w = idf_weight(db_hash)
                    for q_time in query_hashes[db_hash]:
                        matches[song_id].append((q_time, db_time, w))

        # Slope prior
        slope_prior_sigma = 0.15

        def slope_prior(s):
            return np.exp(-0.5 * ((s - 1.0) / slope_prior_sigma) ** 2)

        # Hough search
        slope_candidates = np.arange(0.5, 2.01, 0.02)
        residual_bin_width = 20  # frames (~0.46 seconds)

        scores = {}
        for song_id, tpw in matches.items():
            if len(tpw) < 10:
                continue

            q_t = np.array([p[0] for p in tpw], dtype=float)
            d_t = np.array([p[1] for p in tpw], dtype=float)
            wts = np.array([p[2] for p in tpw], dtype=float)

            best_slope = None
            best_offset = None
            best_final_score = 0
            best_weighted_score = 0
            best_inlier_mask = None
            best_inlier_count = 0

            for slope in slope_candidates:
                residuals = d_t - slope * q_t

                r_min, r_max = residuals.min(), residuals.max()

                if r_max - r_min < residual_bin_width:
                    ws = wts.sum()
                    ni = len(residuals)
                    off = np.median(residuals)
                    mask = np.ones(len(residuals), dtype=bool)
                else:
                    bins = np.arange(r_min, r_max + residual_bin_width,
                                     residual_bin_width)
                    bi = np.digitize(residuals, bins) - 1
                    bi = np.clip(bi, 0, len(bins) - 2)

                    wc = np.zeros(len(bins) - 1)
                    rc = np.zeros(len(bins) - 1, dtype=int)
                    for idx in range(len(residuals)):
                        wc[bi[idx]] += wts[idx]
                        rc[bi[idx]] += 1

                    best_bin = np.argmax(wc)
                    ws = wc[best_bin]
                    ni = rc[best_bin]
                    off = (bins[best_bin] + bins[best_bin + 1]) / 2
                    mask = np.abs(residuals - off) < residual_bin_width

                final = ws * slope_prior(slope)

                if final > best_final_score:
                    best_final_score = final
                    best_weighted_score = ws
                    best_slope = slope
                    best_offset = off
                    best_inlier_mask = mask
                    best_inlier_count = ni

            if best_inlier_count > 0:
                time_pairs = [(p[0], p[1]) for p in tpw]
                n_db_fps = self.database_info.get(
                    song_id, {}
                ).get('n_fingerprints', len(self.database[song_id]))

                normalizer = np.sqrt(n_query_fps * n_db_fps)
                norm_score = (best_final_score / normalizer
                              if normalizer > 0 else 0)

                scores[song_id] = {
                    'score': int(best_inlier_count),
                    'weighted_score': float(best_weighted_score),
                    'prior_penalty': float(slope_prior(best_slope)),
                    'normalized_score': float(norm_score),
                    'total_hash_matches': len(tpw),
                    'tempo_ratio': best_slope,
                    'offset': best_offset,
                    'time_pairs': time_pairs,
                    'inlier_mask': best_inlier_mask,
                    'n_db_fingerprints': n_db_fps
                }

        if not scores:
            print("No matches found!")
            return None, 0

        best_match = max(scores, key=lambda x: scores[x]['normalized_score'])
        best_info = scores[best_match]

        # Leaderboard
        print(f"\n{'Song':<28s} {'Aligned':>7s} {'Wt.Scr':>7s} "
              f"{'Prior':>6s} {'Norm':>9s} {'DB FPs':>7s} {'Slope':>6s}")
        print("-" * 78)
        for sid, info in sorted(scores.items(),
                                key=lambda x: x[1]['normalized_score'],
                                reverse=True):
            m = " ◀" if sid == best_match else ""
            print(f"  {self.song_names[sid]:<26s} "
                  f"{info['score']:>7d} "
                  f"{info['weighted_score']:>7.1f} "
                  f"{info['prior_penalty']:>6.3f} "
                  f"{info['normalized_score']:>9.5f} "
                  f"{info['n_db_fingerprints']:>7d} "
                  f"{info['tempo_ratio']:>6.2f}"
                  f"{m}")

        print(f"\n Best Match: {self.song_names[best_match]}")
        print(f"   Normalized score     : {best_info['normalized_score']:.5f}")
        print(f"   Tempo ratio          : {best_info['tempo_ratio']:.3f}")
        print(f"   Prior penalty        : {best_info['prior_penalty']:.3f}")

        if plot_results:
            self.plot_match_results(best_match, best_info)

        return best_match, best_info['score']

    # 10. Match with separation
    def match_audio_with_separation(self, filepath, plot_results=True):
        vocal_audio = self.separate_vocals(filepath)
        return self.match_audio(vocal_audio, plot_results=plot_results)

    # 11. Plot results
    def plot_match_results(self, song_id, match_info):
        time_pairs = match_info['time_pairs']
        inlier_mask = match_info['inlier_mask']
        slope = match_info['tempo_ratio']
        intercept = match_info['offset']

        if not time_pairs:
            return

        query_times = np.array([p[0] for p in time_pairs])
        db_times = np.array([p[1] for p in time_pairs])

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        ax1.scatter(
            query_times[~inlier_mask], db_times[~inlier_mask],
            alpha=0.2, s=15, c='gray', label='Outliers'
        )
        ax1.scatter(
            query_times[inlier_mask], db_times[inlier_mask],
            alpha=0.7, s=30, c='tab:blue',
            label=f'Inliers ({inlier_mask.sum()})'
        )

        x_range = np.array([query_times.min(), query_times.max()])
        ax1.plot(x_range, slope * x_range + intercept,
                 'r-', linewidth=2,
                 label=f'Fit (slope={slope:.3f})')
        ax1.plot(x_range, x_range + intercept,
                 'g--', linewidth=1, alpha=0.5,
                 label='Slope=1 (same tempo)')

        ax1.set_xlabel('Query Time (frames)')
        ax1.set_ylabel('Database Time (frames)')
        ax1.set_title(f'{len(time_pairs)} matches, '
                      f'{inlier_mask.sum()} inliers')
        ax1.legend()
        ax1.grid(alpha=0.3)

        predicted = slope * query_times + intercept
        residuals = db_times - predicted
        ax2.hist(residuals, bins=80, color='tab:blue',
                 alpha=0.7, edgecolor='black')
        ax2.axvline(0, color='red', linestyle='--',
                    linewidth=1.5, label='Perfect alignment')
        ax2.set_xlabel('Residual (frames)')
        ax2.set_ylabel('Count')
        ax2.set_title('Residual Distribution')
        ax2.legend()
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        if abs(slope - 1.0) < 0.05:
            print(f"  → Tempo nearly identical (ratio = {slope:.3f})")
        elif slope < 1.0:
            print(f"  → Cover SLOWER (ratio = {slope:.3f}, "
                  f"~{slope*100:.1f}% speed)")
        else:
            print(f"  → Cover FASTER (ratio = {slope:.3f}, "
                  f"~{slope*100:.1f}% speed)")

In [ ]:
chroma_shazam = ChromaShazam(sample_rate=22050)

original_songs = [
    ("De La Soul - Breakadawn.mp3", "original_0"),
    ("Fire In The Hole.mp3", "original_1"),
    ("Life On Mars.mp3", "original_2"),
    ("The Girl From Ipanema.mp3", "original_3"),
    ("What You Won't Do for Love.mp3", "original_4")
]

print("=" * 60)
print("BUILDING DATABASE")
print("=" * 60)

for filepath, song_id in original_songs:
    if os.path.exists(filepath):
        chroma_shazam.add_song_to_database_with_separation(
            filepath, song_id=song_id, max_peaks=500
        )
    else:
        print(f"Warning: {filepath} not found — skipping")

print(f"\nDatabase contains {len(chroma_shazam.database)} songs")

# Test
print("\n" + "=" * 60)
print("TESTING COVER DETECTION")
print("=" * 60)

cover_path = "The Girl From Ipanema cover.mp3"

if os.path.exists(cover_path):
    matched, score = chroma_shazam.match_audio_with_separation(
        cover_path, plot_results=True
    )
    if matched:
        print(f"\n Cover matched to: {chroma_shazam.song_names[matched]}")
    else:
        print("\n No match found")
else:
    print(f"Cover file {cover_path} not found")